In [1]:
import numpy as np
import rebound
import matplotlib.pyplot as plt
import os
import pandas as pd
from datetime import datetime

os.chdir('../src/')

from prop_elem import *
from sbdynt import *

# Computing Proper Elements Using Custom Inputs

While many users will be interested in primarily just running the default TNO and asteroid analysis suite provided by the ``run_tno`` and ``run_asteroid`` functions, many users will be interested in computing the proper elements without running the machine learning and chaos indicators. 

In addition, users may want to specify other parameters, such as the length of integration, output cadence, or the planets included in the simulation.
SBDynT allows for custom runs to produce proper elements easily using the underying functions which are called directly by the ``run_tno`` and ``run_asteroid`` functions, which we will describe here. 

Functionality also exists for users to provide their own simulation data, if they have time arrays for the orbital elements of an object from some other integraotr, such as Mercury, SWIFT, or ASSIST. This opens up the computaiton fo proper elements to any users who may wish to comptue proper elements using their own simulation data.

# Initializing Integration

To compute synthetic proper elements, we need to produce an initial REBOUND simulation of the Solar System accompanied by the particles of interest. To initialize the default simulations for TNO's and asteroid, SBDynT provides the ``setup_default_tno_integration`` and ``setup_default_asteroid_integration`` functions. 

This function does not require the small_body class meaning a user may use these functions to initialize objects outisde of the small_body class ecosystem if they wish. 

``initialize_simulation(planets=['all'], des=None, clones=0, cloning_method='Gaussian', datadir='', saveic=False, logfile=False, save_sbdb=False)``
``returns flag, epoch, sim, weights``

The primary inputs required by the user to include will be the ``planets`` and ``des``variables. The ``clones` and ``cloning_methods`` variables are primarily of interest for users wishing the run the machine_learning analysis or some chaos indicator analysis, while the rest of the variables are primarily bookkeeping parameters which are relevant to every analysis.

    (1) planets   (list): 
        Acceptable inputs:
            ['all'] = ['mercury','venus','earth','mars','jupiter','saturn','uranus','neptune']
            (asteroid recommendation) ['inner'] = ['venus','earth','mars','jupiter','saturn','uranus','neptune']
            (tno recommendation) ['outer'] = ['jupiter','saturn','uranus','neptune'] 
            A list containing your choice of ['mercury','venus','earth','mars','jupiter','saturn','uranus','neptune'].
            
            We note that the proper element code requires the planets to be included to compute the proper secular frequencies for the planets themsevles. In the current SBDynT iteration, issues may arise during proper element calculation if you use a custom planets list. 

    (2) des   (str):
        The accepted designation for your small body object as contained within JPL Horizons. Will be used to query Horizons for the orbital elements, and will be the particle hash of the object within the REBOUND simulation.

    (3) datadir (str): 
        path for saving any files produced in this function; defaults to the current directory
        
    (4) saveic (boolean OR str):        
        if True or str:  will save a rebound file with the simulation state that can be used to restart later either to a default file name 'des_init.bin', or to a file with the name equal to the string passed
        if False no initial state is saved
        
    (5) logfile (boolean OR string): 
        if True:  will save some messages to a default log file name or to a file with the name equal to the string passed or to the screen if 'screen' is passed 
        if False no log is saved or output
        
    (6) save_sbdb (boolean OR string): 
        if True:  will save a pickle file with the results of the JPL SBDB query either to a default file name or to a filewith the name equal to the string passed
        if False nothing is saved

Below, we initialize simulations for the asteroid ``Ceres`` and the TNO ``Albion``. using the recommended planets for each type of object. 

In [2]:
flag1, epoch1, sim1 = initialize_simulation(planets=['inner'], des='Ceres', clones=0,  datadir='../example-notebooks/example_sims', saveic=True, logfile=True, save_sbdb=True)
flag2, epoch2, sim2 = initialize_simulation(planets=['outer'], des='Albion', clones=0, datadir='../example-notebooks/example_sims', saveic=True, logfile=True, save_sbdb=True)

We can see that in the exmaple_sims directory, there are now several files that have been populated, including a log file for the current date, a pkl file outling the exact query made to JPL Horizons, and initial condition REBOUND simulations which are equivalent to the sim1 and sim2 simulations we have saved as variables above. 
 

# Running the Integration

Now that our simulations are initialized, we can run the integration using the ``run_simulation`` function.

rflag, sim = integrate_for_pe(sim, des=None, archivefile=None,datadir=None,
                                       logfile=None,tmax=10e6,tout=500., direction='bf')
                                       
    (1) sim  (rebound simulation): Your initial coniditon simulation produced above. 

    (2) tmax (int OR float): The total time in years of simulation you wish to integrate for. In a simulation which is integrated both in the forwards nd backwards direction, the time of integration will span the range (-tmax/2, tmax/2). default = 10 Myr. This is the recommended length of time for asteroid proper element computation. 150 Myr is the recommendaiton for TNOs. 

    (3) tout (int OR float): The output cadence you would like RBEOUND to save snapshots at. default = 500 years, which is the recommend maximum output for asteroids. 5000 years is the maximum recommended output for TNOs. 

    (4) direction ('bf', 'backwards', 'forwards'): The direction of integration you want the simulation to simulate. default = 'bf', which integrates forwards for tmax/2, and then integrates backwards from t=0 for -tmax/2.

We show examples integrating both Ceres and Eris, setting tmax for much shorter timescales, but sufficient to capture the slowest plamnetary secular frequencies.

In [3]:
rflag1, sim1 = integrate_for_pe(sim1, des='Ceres', datadir='../example-notebooks/example_sims', logfile=True,tmax=10e5,tout=500.)
rflag2, sim2 = integrate_for_pe(sim2, des='Albion', datadir='../example-notebooks/example_sims', logfile=True,tmax=150e4,tout=5000.)


# Reading the Simulation

Users familiar with Rebound Simulationarchives may be able to read in the orbital elements from the completed simulations themsevles, but SBDynT provides a useful function for reading in the orbital elements for the planets, the small body, and any included clones, while also sorting the orbital elements by time, and handling complex cases, such as multi-resolution simulations, like those used in the machine learning analyses.

read_archive_for_pe(des, clones=3, datadir='',archivefile='', logfile='', object_type= None)

In [4]:
pr_flag1, time1, sb_elems1, planet_elems1, clone_elems1, small_planets_flag1 =  read_archive_for_pe('Ceres', clones=0, datadir='../example-notebooks/example_sims', logfile=True, object_type= None)
pr_flag2, time2, sb_elems2, planet_elems2, clone_elems2, small_planets_flag2 =  read_archive_for_pe('Albion', clones=0, datadir='../example-notebooks/example_sims', logfile=True, object_type= None)

# Computing Synthetic Proper Elements

To compute synthetic proper elements for the small_body the typical user will simply call the ``compute_proper`` function, as shown below.
The function call will print the comptued proper elements out; additional outputs, such as the uncertianties, are saved to the particle and can be retrieved directly. 

In [5]:
pflag1, pe1 = calc_proper_elements(des='Ceres', times = time1, sb_elems = sb_elems1, planet_elems = planet_elems1, small_planets_flag = small_planets_flag1)
pflag2, pe2 = calc_proper_elements(des='Albion', times = time2, sb_elems = sb_elems2, planet_elems = planet_elems2, small_planets_flag = small_planets_flag2)

In [6]:
print(vars(pe1))

{'planets': ['venus', 'earth', 'mars', 'jupiter', 'saturn', 'uranus', 'neptune'], 'planet_freqs': {'g5': 2.2607394649937922e-06, 'g6': 2.1853814828273325e-05, 'g7': 3.0143192866583898e-06, 'g8': 7.535798216645974e-07, 's6': -1.9593075363279535e-05, 's7': -2.2607394649937922e-06, 's8': -3.7678991083229873e-06, 'g2': 4.5214789299875845e-06, 'g3': 1.3564436789962754e-05, 'g4': 6.782218394981377e-06, 's3': -1.3564436789962754e-05, 's4': -6.782218394981377e-06, 's2': -5.275058751652182e-06}, 'tmax': 0, 'tout': 0, 'osculating_elements': {'a': array([2.76928703]), 'e': array([0.07687512]), 'I': array([0.18485269]), 'o': array([1.28820042]), 'O': array([1.40152026])}, 'proper_elements': {'a': 2.763877354009801, 'e': 0.04880860447537663, 'sinI': 0.07168664671528753, 'g(rev/yr)': 4.2200470013217456e-05, 's(rev/yr)': -4.5968369121540446e-05, 'g("/yr)': 54.69180913712982, 's("/yr)': -59.57500638151642}, 'mean_elements': {'a': 2.763877354009801, 'e': 0.10956004303850146, 'sinI': 0.17173145016337912

In [7]:
print('Secular Plaentary Frequencies: ',pe1.planet_freqs)

print('Proper Elements: ', pe1.proper_elements)
print('Proper Errors: ', pe1.proper_errors)

print('Mean Elements: ', pe1.mean_elements)
print('Osculating Elements: ', pe1.osculating_elements)

Secular Plaentary Frequencies:  {'g5': 2.2607394649937922e-06, 'g6': 2.1853814828273325e-05, 'g7': 3.0143192866583898e-06, 'g8': 7.535798216645974e-07, 's6': -1.9593075363279535e-05, 's7': -2.2607394649937922e-06, 's8': -3.7678991083229873e-06, 'g2': 4.5214789299875845e-06, 'g3': 1.3564436789962754e-05, 'g4': 6.782218394981377e-06, 's3': -1.3564436789962754e-05, 's4': -6.782218394981377e-06, 's2': -5.275058751652182e-06}
Proper Elements:  {'a': 2.763877354009801, 'e': 0.04880860447537663, 'sinI': 0.07168664671528753, 'g(rev/yr)': 4.2200470013217456e-05, 's(rev/yr)': -4.5968369121540446e-05, 'g("/yr)': 54.69180913712982, 's("/yr)': -59.57500638151642}
Proper Errors:  {'RMS_a': 0.0002700602201628314, 'RMS_e': 0.025896022875761447, 'RMS_sinI': 0.038714730549283524}
Mean Elements:  {'a': 2.763877354009801, 'e': 0.10956004303850146, 'sinI': 0.17173145016337912, 'g(rev/yr)': 2.8270096000008492e-05, 's(rev/yr)': -2.7027813069333622e-05, 'g("/yr)': 36.638044416011006, 's("/yr)': -35.0280457378

# Chaos Indicators

Chaos indicators can be produced in much the same way as the proper elements. 3 of the chaos indicators provided by SBDynT can be computed directly from the default small_body integrations for proper elements, namely:
    (1) ACFI
    (2) Entropy
    (3) Power Distribution

The Root Mean Square indciator can also be produced from the integraiton if clones were included in the above integration. 

In addition, if proper elements have already been computed, the Distance Metric chaos indicator cna be generated directly from the values contained within th small_body object. 



Choas indicators can be computed generally using the ``compute_chaos`` function, which will compute choas indciators for every indicator which can be computed form the given simulation. For example, let us look at the process of computing choas indicators with the ``albion`` small_body object below.

In [8]:
albion = sbdynt.small_body('Albion',object_type = 'tno', savefile = True)
albion.compute_chaos()
print(albion.chaos_indicators)

NameError: name 'sbdynt' is not defined

Note that the ceres.chaos_indicators dictionary currently shows empty values for each of the indicators. Running compute_chaos before producing an integration will populate none of the indicators, since nothing can be computed from the present information. 

In [ ]:
albion.init_pe()
albion.integrate(tmax = 1e6)
albion.read_orbel_to_small_body()
albion.compute_chaos()
print(albion.chaos_indicators)

Now that we have run an integration, there is enough data present to comptue the ACFI, Entropy, and Power chaos indicators.

However, since proper elements have not yet been computed, the Distance Metric indicator remains empty.

The Clone RMS indicators are also empty as we did not integrate any clones. Let us do so now.

In [ ]:
albion = sbdynt.small_body('Albion',object_type = 'tno', savefile = True, clones=2)
albion.init_pe()
albion.integrate(tmax = 1e6)
albion.read_orbel_to_small_body()
albion.compute_chaos()
print(albion.chaos_indicators)

In [ ]:
albion.compute_proper()
albion.compute_chaos()
print(albion.chaos_indicators)